#**Practical No. 6**

**Aim:** To configure a Directed Acyclic Graph (DAG) in Apache Airflow to run daily and execute sequential dependency tasks consisting of pipeline initialization, an extraction script, and pipeline success logging

In [1]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
!pip install "apache-airflow==3.3.2" \
    --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-3.3.2/constraints-3.13.txt"

In [3]:
!airflow version

3.3.2


In [4]:
import os

os.environ["AIRFLOW_HOME"] = "/content/airflow"

print("AIRFLOW_HOME =", os.environ["AIRFLOW_HOME"])

AIRFLOW_HOME = /content/airflow


In [5]:
import os

os.makedirs("/content/airflow/dags", exist_ok=True)

print("DAG directory created:")
print("/content/airflow/dags")

DAG directory created:
/content/airflow/dags


In [6]:
dag_code = r'''
from datetime import datetime

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator


# ---------------------------------------------------------
# Task 1: Pipeline Initialization
# ---------------------------------------------------------

def initialize_pipeline():
    print("========================================")
    print("PIPELINE INITIALIZATION")
    print("========================================")
    print("Pipeline initialization started.")
    print("Pipeline is ready to execute.")
    print("Initialization completed successfully.")


# ---------------------------------------------------------
# Task 2: Data Extraction
# ---------------------------------------------------------

def extract_data():
    print("========================================")
    print("DATA EXTRACTION")
    print("========================================")
    print("Data extraction started.")

    # Demonstration extraction process
    data = [
        {"id": 1, "name": "Alice"},
        {"id": 2, "name": "Bob"},
        {"id": 3, "name": "Charlie"}
    ]

    print("Extracting data from source...")
    print("Records extracted:", len(data))
    print("Data extraction completed successfully.")


# ---------------------------------------------------------
# Task 3: Pipeline Success Logging
# ---------------------------------------------------------

def pipeline_success():
    print("========================================")
    print("PIPELINE SUCCESS")
    print("========================================")
    print("All pipeline tasks completed successfully.")
    print("Pipeline execution successful.")


# ---------------------------------------------------------
# DAG Configuration
# ---------------------------------------------------------

with DAG(
    dag_id="data_engineering_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule="@daily",
    catchup=False,
    description="Daily Data Engineering pipeline",
) as dag:

    initialize = PythonOperator(
        task_id="pipeline_initialization",
        python_callable=initialize_pipeline,
    )

    extraction = PythonOperator(
        task_id="data_extraction",
        python_callable=extract_data,
    )

    success = PythonOperator(
        task_id="pipeline_success_logging",
        python_callable=pipeline_success,
    )

    # Sequential dependency
    initialize >> extraction >> success
'''

with open("/content/airflow/dags/data_pipeline_dag.py", "w") as f:
    f.write(dag_code)

print("DAG file created successfully.")
print("/content/airflow/dags/data_pipeline_dag.py")

DAG file created successfully.
/content/airflow/dags/data_pipeline_dag.py


In [7]:
!airflow db migrate

2026-09-27T18:43:45.056460Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////content/airflow/airflow.db
2026-09-27T18:43:45.208435Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-27T18:43:45.208742Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-27T18:43:45.210745Z [info     ] Migrating the Airflow database [airflow.utils.db] loc=db.py:1189
2026-09-27T18:43:45.220256Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-27T18:43:45.220513Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-27T18:43:45.265581Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-27T18:43:45.265905Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] lo

In [8]:
!airflow dags list

              |              |         |           |             | bundle_versio
dag_id        | fileloc      | owners  | is_paused | bundle_name | n            
==============+==============+=========+===========+=============+==============
data_engineer | /content/air | airflow | True      | dags-folder | None         
ing_pipeline  | flow/dags/da |         |           |             |              
              | ta_pipeline_ |         |           |             |              
              | dag.py       |         |           |             |              
                                                                                


In [17]:
!airflow tasks list data_engineering_pipeline

data_extraction
pipeline_initialization
pipeline_success_logging


In [9]:
!airflow dags show data_engineering_pipeline

digraph data_engineering_pipeline {
	graph [label=data_engineering_pipeline labelloc=t rankdir=LR]
	data_extraction [color="#000000" fillcolor="#ffefeb" label=data_extraction shape=rectangle style="filled,rounded"]
	pipeline_initialization [color="#000000" fillcolor="#ffefeb" label=pipeline_initialization shape=rectangle style="filled,rounded"]
	pipeline_success_logging [color="#000000" fillcolor="#ffefeb" label=pipeline_success_logging shape=rectangle style="filled,rounded"]
	data_extraction -> pipeline_success_logging
	pipeline_initialization -> data_extraction
}



In [10]:
!airflow dags test data_engineering_pipeline 2026-09-27

2026-09-27T18:44:22.008706Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-09-27T18:44:22.024574Z [info     ] Filling up the DagBag from /content/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:460
2026-09-27T18:44:22.064817Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:218
2026-09-27T18:44:22.097594Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:453 next_info=DagRunInfo(run_after=DateTime(2026, 9, 27, 0, 0, 0, tzinfo=Timezone('UTC')), data_interval=DataInterval(start=DateTime(2026, 9, 27, 0, 0, 0, tzinfo=Timezone('UTC')), end=DateTime(2026, 9, 27, 0, 0, 0, tzinfo=Timezone('UTC'))), partition_date=None, partition_key=None)
2026-09-27T18:44:22